In [ ]:
# Parameters
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "tests").exists():
            return candidate
    raise RuntimeError(f"Unable to locate repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
EXPERIMENT_CONFIG_PATH = str(
    REPO_ROOT
    / "tests"
    / "nb_experiments"
    / "concept_direction"
    / "analysis"
    / "configs"
    / "gemma3_1b_it_latent_dynamics_color_fruit_orange_fs_l10_n5.yaml"
)
PROJECTION_METHOD = "umap"
STAGE_TOP_N = 5

# Concept Direction Latent Dynamics Analysis

This notebook inspects the intermediate latent states behind the `gemma-3-1b-it` concept-direction pipeline for one preserved surface. It can be run manually or via papermill by overriding `EXPERIMENT_CONFIG_PATH`, `PROJECTION_METHOD`, and `STAGE_TOP_N`.

The default config extends the orange `fs_l10_n5` surface so the stage-wise top features and projection plots stay aligned with the later-layer parity artifacts.

The report now separates the mean-difference proto-directions from the paired-rejection store directions, exposes normalized direction hashes, and attaches Neuronpedia feature links plus best-available explanation text. `selected_store_state_source` shows whether the final store state came from the projected context state or from the answer-state fallback.

In [ ]:
import torch  # noqa: E402

if torch.cuda.is_available():
    torch.cuda.init()
    device = torch.device("cuda")
    with torch.cuda.device(device):
        warmup = torch.ones((1, 1), device=device, dtype=torch.float32)
        _ = warmup @ warmup
        torch.cuda.synchronize(device)

In [ ]:
import html  # noqa: E402
import importlib  # noqa: E402
import sys  # noqa: E402
from pathlib import Path  # noqa: E402

import pandas as pd  # noqa: E402
import plotly.express as px  # noqa: E402
from IPython.display import HTML, Markdown, display  # noqa: E402


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "tests").exists():
            return candidate
    raise RuntimeError(f"Unable to locate repo root from {start}")


def _as_html_link(url: object, label: object) -> str:
    if label is None or _is_missing(label):
        return "N/A"
    label_text = html.escape(str(label))
    if not isinstance(url, str) or not url:
        return label_text
    return f'<a href="{html.escape(url, quote=True)}" target="_blank">{label_text}</a>'


def _is_missing(value: object) -> bool:
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def _sanitize_hover_value(value: object) -> object:
    return "N/A" if _is_missing(value) else value


def _sanitize_table_value(value: object) -> object:
    return "N/A" if _is_missing(value) else value


def _prepare_hover_frame(
    frame: pd.DataFrame,
    *,
    hover_fields: list[str] | None,
    hover_name: str | None,
) -> pd.DataFrame:
    plot_frame = frame.copy()
    required_fields = list(dict.fromkeys([*(hover_fields or []), *([hover_name] if hover_name else [])]))
    for field in required_fields:
        if field not in plot_frame.columns:
            plot_frame[field] = "N/A"
            continue
        plot_frame[field] = plot_frame[field].map(_sanitize_hover_value)
    if "hover_label" in plot_frame.columns:
        plot_frame["hover_label"] = plot_frame["hover_label"].map(_sanitize_hover_value)
    return plot_frame


def _feature_id_link(row: pd.Series) -> str:
    return _as_html_link(row.get("feature_url"), row.get("feature_id"))


def _format_token_label_html(rows: object) -> str:
    if not isinstance(rows, list):
        return ""
    return format_token_debug_html(
        rows,
        token_text_key="token_label",
        marks_key=None,
        index_key=None,
    )


def _display_html_table(
    frame: pd.DataFrame,
    *,
    html_column_names: list[str] | None = None,
    max_height_px: int | None = None,
) -> None:
    html_column_set = set(html_column_names or [])
    safe_frame = frame.copy()
    for column in safe_frame.columns:
        if column in html_column_set:
            continue
        safe_frame[column] = safe_frame[column].map(_sanitize_table_value)
        safe_frame[column] = safe_frame[column].map(
            lambda value: html.escape(str(value)) if isinstance(value, str) else value
        )
    table_html = safe_frame.to_html(index=False, escape=False)
    if max_height_px is not None:
        table_html = (
            f'<div style="max-height: {int(max_height_px)}px; overflow-y: auto; overflow-x: auto;">{table_html}</div>'
        )
    display(HTML(table_html))


def display_table(
    frame: pd.DataFrame,
    *,
    hide_columns: list[str] | None = None,
    html_columns: dict[str, object] | None = None,
    max_height_px: int | None = None,
) -> None:
    if frame.empty:
        display(Markdown("_No rows produced._"))
        return

    display_frame = frame.copy()
    for field_name, formatter in (html_columns or {}).items():
        if field_name not in display_frame.columns:
            continue
        display_frame[field_name] = display_frame.apply(lambda row: formatter(row), axis=1)
    for field in hide_columns or []:
        if field in display_frame.columns:
            display_frame = display_frame.drop(columns=field)
    plain_columns = [field for field in display_frame.columns if field not in (html_columns or {})]
    for field in plain_columns:
        display_frame[field] = display_frame[field].map(_sanitize_table_value)
    if html_columns:
        _display_html_table(
            display_frame,
            html_column_names=list(html_columns),
            max_height_px=max_height_px,
        )
        return
    display_full_frame(display_frame)


def display_projection(
    frame: pd.DataFrame,
    *,
    title: str,
    color: str | None = None,
    symbol: str | None = None,
    hover_fields: list[str] | None = None,
    text: str | None = None,
    hover_name: str | None = None,
    hide_table_columns: list[str] | None = None,
    html_columns: dict[str, object] | None = None,
    height: int | None = None,
    width: int | None = None,
    table_max_height_px: int | None = None,
) -> None:
    if frame.empty:
        display(Markdown(f"_{title}: no rows produced._"))
        return

    plot_frame = _prepare_hover_frame(
        frame,
        hover_fields=hover_fields,
        hover_name=hover_name,
    )
    resolved_hover_name = None
    if hover_name and hover_name in plot_frame.columns:
        resolved_hover_name = hover_name
    elif "hover_label" in plot_frame.columns:
        resolved_hover_name = "hover_label"

    resolved_hover_fields = [
        field
        for field in (hover_fields or [])
        if field in plot_frame.columns and field not in {"x", "y", resolved_hover_name}
    ]

    plot_kwargs = {
        "data_frame": plot_frame,
        "x": "x",
        "y": "y",
        "title": title,
        "hover_name": resolved_hover_name,
    }
    if resolved_hover_fields:
        plot_kwargs["hover_data"] = {field: True for field in resolved_hover_fields}
    if color and color in plot_frame.columns:
        plot_kwargs["color"] = color
    if symbol and symbol in plot_frame.columns:
        plot_kwargs["symbol"] = symbol
    if text and text in plot_frame.columns:
        plot_kwargs["text"] = text

    fig = px.scatter(**plot_kwargs)
    trace_updates = {"marker": dict(size=9, opacity=0.9)}
    if text and text in plot_frame.columns:
        trace_updates.update({"textposition": "top center", "cliponaxis": False})
    fig.update_traces(**trace_updates)
    layout_updates = {}
    if height is not None:
        layout_updates["height"] = height
    if width is not None:
        layout_updates["width"] = width
    fig.update_layout(margin=dict(l=120, r=120, t=90, b=90))
    if layout_updates:
        fig.update_layout(**layout_updates)
    fig.show()

    display_table(
        frame,
        hide_columns=hide_table_columns,
        html_columns=html_columns,
        max_height_px=table_max_height_px,
    )


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tests.nb_experiments.nb_harness_utils import (  # noqa: E402
    display_full_frame,
    format_token_debug_html,
)

latent_dynamics = importlib.import_module(
    "tests.nb_experiments.concept_direction.analysis.concept_direction_latent_dynamics"
)
latent_dynamics = importlib.reload(latent_dynamics)
run_latent_dynamics_analysis = latent_dynamics.run_latent_dynamics_analysis
build_latent_dynamics_frames = latent_dynamics.build_latent_dynamics_frames

In [ ]:
report = run_latent_dynamics_analysis(
    EXPERIMENT_CONFIG_PATH,
    projection_method=PROJECTION_METHOD,
    stage_top_n=STAGE_TOP_N,
)
frames = build_latent_dynamics_frames(report)
report["config"]

In [ ]:
stage_summary = frames["stage_summary"].copy()
if not stage_summary.empty:
    stage_summary["normalized_direction_sha"] = stage_summary["normalized_direction_sha"].astype(str).str.slice(0, 8)

print("RESOLVED_KEY_TOKENS")
print(frames["config"][["resolved_key_tokens"]].to_string(index=False))

print("\nSTAGE_SUMMARY")
summary_stage_columns = [
    "stage_rank",
    "stage_name",
    "stage_kind",
    "direction_norm",
    "raw_direction_norm",
    "latent_row_count",
    "top_feature_count",
    "normalized_direction_sha",
]
print(stage_summary[summary_stage_columns].to_string(index=False))

print("\nGROUP_MEAN_STATE_NORMS")
group_means = (
    frames["examples"]
    .groupby(["group_name", "selected_store_state_source"])[
        [
            "answer_state_norm",
            "context_state_norm",
            "projected_context_state_norm",
            "selected_store_state_norm",
            "projected_selected_state_delta_norm",
            "answer_context_cosine",
            "expected_answer_logit_margin",
        ]
    ]
    .mean()
    .reset_index()
)
print(group_means.to_string(index=False))

print("\nEXPECTED_ANSWER_LOGIT_MARGINS")
print(
    frames["examples"][
        [
            "group_name",
            "probe_surface_text",
            "expected_answer",
            "logit_diff_label",
            "logit_diff",
            "expected_answer_logit_margin",
        ]
    ].to_string(index=False)
)

print("\nPAIR_RESIDUALS")
print(
    frames["pair_residuals"][
        ["pair_index", "pair_weight", "projection_norm", "residual_norm", "group_cosine"]
    ].to_string(index=False)
)

print("\nFEATURE_VECTOR_COUNTS")
if frames["feature_vectors"].empty:
    print("<no feature vectors resolved>")
else:
    feature_vector_summary = (
        frames["feature_vectors"].groupby(["stage_name", "vector_type"]).size().reset_index(name="count")
    )
    print(feature_vector_summary.to_string(index=False))

print("\nKEY_STAGE_TOP_FEATURES")
for stage_name in [
    "target_token_embed_difference_direction",
    "projected_context_state_mean_difference_direction",
    "paired_residual_mean_direction_unweighted",
    "store_context_enhanced_paired_rejection_unnormalized_direction",
    "store_context_enhanced_paired_rejection_direction",
    "store_answer_position_paired_rejection_direction",
]:
    stage_frame = frames["stage_features"].loc[frames["stage_features"]["stage_name"] == stage_name]
    print(f"[{stage_name}]")
    if stage_frame.empty:
        print("<no feature rows>")
    else:
        stage_columns = [
            "rank",
            "layer",
            "position",
            "feature_id",
            "score",
            "feature_explanation",
        ]
        print(stage_frame[stage_columns].to_string(index=False))
    print()

## Structured Tables

### Overview Tables

In [ ]:
display_table(frames["config"])
display_table(frames["stage_summary"])
display_table(frames["stage_key_tokens"])

### Feature Vectors

In [ ]:
display_table(
    frames["feature_vectors"],
    hide_columns=["feature_url", "hover_label"],
    html_columns={
        "feature_id": _feature_id_link,
    },
)

### Example Weights And Store-State Metrics

In [ ]:
display(
    Markdown(
        "`selected_store_state_source` shows whether the final store state came from "
        "the projected context state or from the answer-state fallback. "
        "`projected_selected_state_delta_norm = 0` means the selected store state "
        "is numerically identical to the projected context state for that example."
    )
)

### Stage Top Features

In [ ]:
display_table(
    frames["stage_features"],
    hide_columns=["feature_url", "key_tokens"],
    html_columns={
        "feature_id": _feature_id_link,
    },
)

## Projections

### Trajectory Projection

In [ ]:
trajectory_frame = frames["trajectory_projection"]
trajectory_meta = report["trajectory_projection"]
display_projection(
    trajectory_frame,
    title=f"Per-example latent-state trajectory projection ({trajectory_meta['backend']})",
    color="stage_name",
    symbol="group_name",
    hover_fields=[
        "example_index",
        "stage_name",
        "group_name",
        "probe_surface_text",
        "expected_answer",
        "selected_store_state_source",
        "expected_answer_logit_margin",
    ],
)

### Stage Direction Projection

In [ ]:
direction_frame = frames["direction_projection"]
direction_meta = report["direction_projection"]
display_projection(
    direction_frame,
    title=f"Stage direction projection ({direction_meta['backend']})",
    color="stage_name",
    symbol="stage_kind",
    hover_fields=["stage_name", "stage_rank", "stage_kind"],
    text="stage_name",
    height=1020,
    width=1800,
)

### Feature Vector Projection

In [ ]:
feature_vector_frame = frames["feature_vector_projection"].copy()
feature_vector_meta = report["feature_vector_projection"]
feature_vector_frame["feature_explanation_hover"] = feature_vector_frame["feature_explanation"].map(
    lambda value: value if not _is_missing(value) and str(value).strip() else "No explanation available"
)
display_projection(
    feature_vector_frame,
    title=f"Encoder and decoder feature vector projection ({feature_vector_meta['backend']})",
    color="stage_name",
    symbol="vector_type",
    hover_name="feature_explanation_hover",
    hover_fields=[
        "stage_name",
        "stage_rank",
        "rank",
        "layer",
        "position",
        "feature_id",
        "score",
        "vector_type",
    ],
    hide_table_columns=["feature_url", "feature_explanation_hover"],
    html_columns={
        "feature_id": _feature_id_link,
    },
    height=900,
    width=1500,
)

### Aggregate Projection

In [ ]:
aggregate_frame = frames["aggregate_projection"].copy()
aggregate_meta = report["aggregate_projection"]
aggregate_frame["feature_explanation_hover"] = aggregate_frame.apply(
    lambda row: (
        row.get("feature_explanation")
        if not _is_missing(row.get("feature_explanation")) and str(row.get("feature_explanation")).strip()
        else (
            "No explanation available"
            if str(row.get("aggregate_kind", "")).startswith("feature_")
            else row.get("aggregate_kind", "N/A")
        )
    ),
    axis=1,
)
display_projection(
    aggregate_frame,
    title=f"Aggregate latent-state and direction projection ({aggregate_meta['backend']})",
    color="aggregate_kind",
    hover_name="feature_explanation_hover",
    hover_fields=[
        "aggregate_kind",
        "stage_name",
        "stage_rank",
        "group_name",
        "vector_type",
        "feature_id",
        "probe_surface_text",
        "selected_store_state_source",
    ],
    hide_table_columns=["feature_url", "feature_explanation_hover"],
    html_columns={
        "feature_id": _feature_id_link,
    },
    table_max_height_px=440,
)

### Raw Payload Keys

In [ ]:
display(sorted(report.keys()))